# Preparing the data

## Define parameters


In [1]:
from sklearn.svm import SVR
kernels = ['linear','‘poly', 'rbf', 'sigmoid']
C_values = [0.2, 0.4, 0.6, 0.8, 1.0]
rbf_gamma = [0.05, 0.2, 0.4, 0.6]
ps_coef0 = [0.0, 0.05, 0.1, 0.15, 0.2, 0.5]
poly_degree = [1, 2, 3, 5]
train_set_size = 5000

## Load the data

In [2]:
# Load the data

from pathlib import Path
import pandas as pd
import tarfile
import urllib.request
from sklearn.model_selection import train_test_split

def load_housing_data():
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as housing_tarball:
            housing_tarball.extractall(path="datasets", filter="data")
    return pd.read_csv(Path("datasets/housing/housing.csv"))

housing_full = load_housing_data()


### Create train and test set

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler, StandardScaler


train_set, test_set = train_test_split(housing_full, test_size=0.2,
                                       random_state=2911)
train_set = train_set[:train_set_size]

train_set.head(8)
train_labels = train_set["median_house_value"].copy()
train_set.drop("median_house_value", axis=1)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
14333,-117.91,33.76,22.0,7531.0,1569.0,5254.0,1523.0,3.8506,<1H OCEAN
11711,-117.31,34.15,7.0,5747.0,1307.0,2578.0,1147.0,3.3281,INLAND
2424,-121.92,36.62,52.0,1410.0,303.0,578.0,285.0,2.5625,NEAR OCEAN
1972,-118.21,34.64,16.0,2573.0,427.0,1273.0,426.0,5.9508,INLAND
14697,-117.99,33.92,27.0,5805.0,1152.0,3106.0,1144.0,4.0610,<1H OCEAN
...,...,...,...,...,...,...,...,...,...
13126,-122.43,37.74,52.0,2229.0,498.0,1079.0,472.0,5.0196,NEAR BAY
6432,-121.89,37.49,9.0,4909.0,577.0,1981.0,591.0,9.7194,<1H OCEAN
181,-118.10,34.01,29.0,2077.0,564.0,2087.0,543.0,2.6600,<1H OCEAN
17835,-121.57,39.48,15.0,202.0,54.0,145.0,40.0,0.8252,INLAND


In [4]:
# cat_encoder = OneHotEncoder(sparse_output=False)
# housing_cat = train_set[["ocean_proximity"]]

# housing_cat_1hot = pd.DataFrame(cat_encoder.fit_transform(housing_cat),
#                          columns=cat_encoder.get_feature_names_out(),
#                          index=housing_cat.index)
# # print(cat_encoder.feature_names_in_)
# # print(cat_encoder.get_feature_names_out())


# # Drop the original 'ocean_proximity' column from train_set
# train_set_numerical = train_set.drop("ocean_proximity", axis=1)

In [5]:
# Concatenate the numerical features with the one-hot encoded features
# train_set = pd.concat([train_set_numerical, housing_cat_1hot], axis=1)
# print(train_set.info())
# print(train_set.describe())
# std_scaler = StandardScaler()
# std_scaler.set_output(transform="pandas")

# housing_num_std_scaled = std_scaler.fit_transform(train_set_numerical)
# housing_num_std_scaled = pd.concat([housing_num_std_scaled, housing_cat_1hot], axis = 1)
# housing_num_std_scaled.head(8)

# Prepare the models

## Define ClusterSimilarity

In [6]:
from sklearn.cluster import KMeans
from sklearn.base import BaseEstimator, TransformerMixin
# from sklearn.kernel_approximation import RBFSampler
from sklearn.metrics.pairwise import rbf_kernel
class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self  # always return self!

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)

    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]

## Define log pipeline
log pipeline will transform the data to log, and perform a standard scaler on it.


In [7]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer


def column_ratio(X):
    return X[:, [0]] / X[:, [1]]

def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]  # feature names out

def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler())

log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log, feature_names_out="one-to-one"),
    StandardScaler())

cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1., random_state=42)
default_num_pipeline = make_pipeline(SimpleImputer(strategy="median"),
                                     StandardScaler())




## Perform column transformation
separate the numerical and categorical attributes, define some extra columns, like rational ones, and also cluster similarity for geographical features.

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import make_column_selector, make_column_transformer

num_attribs = ["longitude", "latitude", "housing_median_age", "total_rooms",
               "total_bedrooms", "population", "households", "median_income"]
cat_attribs = ["ocean_proximity"]

cat_pipeline = Pipeline([
    ("impute_mf", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore"))
    ])



num_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("standardize", StandardScaler()),
])
preprocessing = ColumnTransformer([
        ("bedrooms", ratio_pipeline(), ["total_bedrooms", "total_rooms"]),
        ("rooms_per_house", ratio_pipeline(), ["total_rooms", "households"]),
        ("people_per_house", ratio_pipeline(), ["population", "households"]),
        ("log", log_pipeline, ["total_bedrooms", "total_rooms", "population",
                               "households", "median_income"]),
        ("geo", cluster_simil, ["latitude", "longitude"]),
        ("cat", cat_pipeline, make_column_selector(dtype_include=object)),
    ],
    remainder=default_num_pipeline)

# preprocessing.set_output(transform="pandas")
housing_prepared = preprocessing.fit_transform(train_set)
housing_prepared_fr = pd.DataFrame(housing_prepared, columns=preprocessing.get_feature_names_out(),index=train_set.index)
housing_prepared_fr.columns

Index(['bedrooms__ratio', 'rooms_per_house__ratio', 'people_per_house__ratio',
       'log__total_bedrooms', 'log__total_rooms', 'log__population',
       'log__households', 'log__median_income', 'geo__Cluster 0 similarity',
       'geo__Cluster 1 similarity', 'geo__Cluster 2 similarity',
       'geo__Cluster 3 similarity', 'geo__Cluster 4 similarity',
       'geo__Cluster 5 similarity', 'geo__Cluster 6 similarity',
       'geo__Cluster 7 similarity', 'geo__Cluster 8 similarity',
       'geo__Cluster 9 similarity', 'cat__ocean_proximity_<1H OCEAN',
       'cat__ocean_proximity_INLAND', 'cat__ocean_proximity_ISLAND',
       'cat__ocean_proximity_NEAR BAY', 'cat__ocean_proximity_NEAR OCEAN',
       'remainder__housing_median_age', 'remainder__median_house_value'],
      dtype='str')

# Define pipeline


In [9]:
from sklearn.svm import SVR

def add_prefix(step_name, params: dict) -> dict:
    return {f"{step_name}__{k}": v for k, v in params.items()}




svr = SVR()
full_pipeline = Pipeline([
      ("preprocessing", preprocessing),
      ("svr", svr)
])
# full_pipeline.get_params()

# Select and train a model


In [ ]:
train_set.head(8)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
14333,-117.91,33.76,22.0,7531.0,1569.0,5254.0,1523.0,3.8506,167400.0,<1H OCEAN
11711,-117.31,34.15,7.0,5747.0,1307.0,2578.0,1147.0,3.3281,122200.0,INLAND
2424,-121.92,36.62,52.0,1410.0,303.0,578.0,285.0,2.5625,235400.0,NEAR OCEAN
1972,-118.21,34.64,16.0,2573.0,427.0,1273.0,426.0,5.9508,181100.0,INLAND
14697,-117.99,33.92,27.0,5805.0,1152.0,3106.0,1144.0,4.0610,222700.0,<1H OCEAN
4277,-122.42,37.66,36.0,725.0,121.0,335.0,140.0,4.1250,327600.0,NEAR OCEAN
17413,-118.01,33.91,32.0,2722.0,571.0,2541.0,462.0,4.2305,221400.0,<1H OCEAN
15425,-120.95,37.61,17.0,4054.0,654.0,2034.0,667.0,4.6833,142200.0,INLAND


## Perform gridsearch(Exercise 1)

In [ ]:
from sklearn.model_selection import GridSearchCV

# kernels = ['linear','‘poly', 'rbf', 'sigmoid']
# C_values = [0.2, 0.4, 0,6, 0.8, 1.0]
# rbf_gamma = [0.05, 0.2, 0.4, 0.6]
# ps_coef0 = [0.0, 0.05, 0.1, 0.15, 0.2, 0.5]
# poly_degree = [1, 2, 3, 5]


param_grid = [
   {
       "svr__kernel": ["linear"],
       "svr__C": C_values
   },
   {
      "svr__kernel": ["poly", "sigmoid"],
      "svr__coef0": ps_coef0,
      "svr__degree": poly_degree,
      "svr__C": C_values
   },
   {
       "svr__kernel": ["rbf"],
       "svr__gamma": rbf_gamma,
       "svr__C": C_values
   }
]

# param_grid = [add_prefix("svr", param) for param in base_grid]





grid_search = GridSearchCV(
    full_pipeline,
    param_grid,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_search.fit(train_set, train_labels)

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(remainder=Pipeline(steps=[('simpleimputer',
                                                                                     SimpleImputer(strategy='median')),
                                                                                    ('standardscaler',
                                                                                     StandardScaler())]),
                                                          transformers=[('bedrooms',
                                                                         Pipeline(steps=[('simpleimputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('functiontransformer',
                                                                                          FunctionTransformer(feature_names_out=<f...
                                       ('svr', SVR())]),
             n_jobs=-1,
             param_grid=[{'svr__C': [0.2, 0.4, 0.6, 0.8, 1.0],
                          'svr__kernel': ['linear']},
                         {'svr__C': [0.2, 0.4, 0.6, 0.8, 1.0],
                          'svr__coef0': [0.0, 0.05, 0.1, 0.15, 0.2, 0.5],
                          'svr__degree': [1, 2, 3, 5],
                          'svr__kernel': ['poly', 'sigmoid']},
                         {'svr__C': [0.2, 0.4, 0.6, 0.8, 1.0],
                          'svr__gamma': [0.05, 0.2, 0.4, 0.6],
                          'svr__kernel': ['rbf']}],
             scoring='neg_root_mean_squared_error')

In [ ]:
results = pd.DataFrame(grid_search.cv_results_)
results["rmse"] = np.sqrt(-results["mean_test_score"])

results.sort_values("rmse")[:10]

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_svr__C,param_svr__kernel,param_svr__coef0,param_svr__degree,param_svr__gamma,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score,rmse
4,1.450760,0.319134,0.528097,0.017791,1.0,linear,NaN,NaN,NaN,"{'svr__C': 1.0, 'svr__kernel': 'linear'}",-113697.356993,-116237.923909,-113108.097958,-114347.792953,1358.001675,1,338.153505
3,1.877805,0.290377,0.509399,0.066167,0.8,linear,NaN,NaN,NaN,"{'svr__C': 0.8, 'svr__kernel': 'linear'}",-114551.313150,-116608.684265,-113925.680763,-115028.559393,1146.138424,2,339.158605
2,2.071460,0.411351,0.607146,0.066133,0.6,linear,NaN,NaN,NaN,"{'svr__C': 0.6, 'svr__kernel': 'linear'}",-115384.387307,-117133.221143,-114765.728528,-115761.112326,1002.562259,3,340.236847
1,2.511224,0.536432,0.777422,0.058984,0.4,linear,NaN,NaN,NaN,"{'svr__C': 0.4, 'svr__kernel': 'linear'}",-116232.420431,-117788.000316,-115557.996589,-116526.139112,933.785183,4,341.359252
0,2.058806,0.516065,0.614316,0.319893,0.2,linear,NaN,NaN,NaN,"{'svr__C': 0.2, 'svr__kernel': 'linear'}",-117123.705914,-118537.045754,-116397.031905,-117352.594524,888.522112,5,342.567650
213,1.175508,0.032682,0.375175,0.005722,1.0,poly,0.10,1.0,NaN,"{'svr__C': 1.0, 'svr__coef0': 0.1, 'svr__degre...",-117623.800923,-119026.823436,-116883.837205,-117844.820521,888.719957,6,343.285334
229,1.184647,0.029769,0.380712,0.028787,1.0,poly,0.20,1.0,NaN,"{'svr__C': 1.0, 'svr__coef0': 0.2, 'svr__degre...",-117623.800923,-119026.823436,-116883.837205,-117844.820521,888.719957,7,343.285334
197,1.162455,0.035924,0.386780,0.005636,1.0,poly,0.00,1.0,NaN,"{'svr__C': 1.0, 'svr__coef0': 0.0, 'svr__degre...",-117623.800923,-119026.823436,-116883.837205,-117844.820521,888.719957,8,343.285334
205,1.175600,0.031613,0.389113,0.019578,1.0,poly,0.05,1.0,NaN,"{'svr__C': 1.0, 'svr__coef0': 0.05, 'svr__degr...",-117623.800923,-119026.823436,-116883.837205,-117844.820521,888.719957,9,343.285334
221,1.195871,0.026569,0.363539,0.009722,1.0,poly,0.15,1.0,NaN,"{'svr__C': 1.0, 'svr__coef0': 0.15, 'svr__degr...",-117623.800924,-119026.823436,-116883.837205,-117844.820521,888.719957,10,343.285334


## Perform RnadomizedSearch(Exercise 2)

In [10]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVR
from scipy.stats import randint, uniform

# kernels = ['linear','‘poly', 'rbf', 'sigmoid']
# C_values = [0.2, 0.4, 0,6, 0.8, 1.0]
# rbf_gamma = [0.05, 0.2, 0.4, 0.6]
# ps_coef0 = [0.0, 0.05, 0.1, 0.15, 0.2, 0.5]
# poly_degree = [1, 2, 3, 5]
C_dist = uniform(0.0, 1.0)
ps_coef0_dist = uniform(0.0, 1.0)
rbf_gamma_dist = uniform(0.0, 1.0)
param_distrs = [
   {
       "svr__kernel": ["linear"],
       "svr__C" : C_dist
   },
   {
      "svr__kernel": ["poly", "sigmoid"],
      "svr__coef0": ps_coef0_dist,
      "svr__degree": randint(1, 5),
      "svr__C": C_dist
   },
   {
       "svr__kernel": ["rbf"],
       "svr__gamma": rbf_gamma_dist,
       "svr__C": C_dist
   }
]

svr = SVR()

random_search = RandomizedSearchCV(full_pipeline,
                                   param_distrs,
                                   cv=3,
                                   scoring="neg_root_mean_squared_error",)

In [11]:
rnd_search = random_search.fit(train_set, train_labels)

In [12]:
cv_res = pd.DataFrame(rnd_search.cv_results_)
cv_res["rmse"] = np.sqrt(-cv_res["mean_test_score"])

cv_res.sort_values("rmse")[:10]

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_svr__C,param_svr__coef0,param_svr__degree,param_svr__kernel,param_svr__gamma,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score,rmse
9,0.465661,0.015847,0.149070,0.009738,0.884685,NaN,NaN,linear,NaN,"{'svr__C': 0.884685401485819, 'svr__kernel': '...",-114187.665825,-1.164386e+05,-113593.402600,-1.147399e+05,1.225414e+03,1,338.732759
3,0.490155,0.022275,0.145951,0.000474,0.706898,NaN,NaN,linear,NaN,"{'svr__C': 0.7068983704522867, 'svr__kernel': ...",-114938.828206,-1.168320e+05,-114309.400146,-1.153601e+05,1.072048e+03,2,339.646976
1,0.563214,0.030732,0.197366,0.012162,0.533562,NaN,NaN,linear,NaN,"{'svr__C': 0.5335618832595489, 'svr__kernel': ...",-115660.508852,-1.173509e+05,-115045.392611,-1.160189e+05,9.747287e+02,3,340.615504
6,0.904951,0.038153,0.334039,0.005998,0.445396,0.482287,1.0,sigmoid,NaN,"{'svr__C': 0.44539598580039297, 'svr__coef0': ...",-117897.237951,-1.192877e+05,-117121.873538,-1.181023e+05,8.960000e+02,4,343.660107
8,0.482943,0.006778,0.154732,0.003997,0.284690,0.230227,1.0,poly,NaN,"{'svr__C': 0.2846897452551599, 'svr__coef0': 0...",-117909.962756,-1.192947e+05,-117131.783786,-1.181121e+05,8.944888e+02,5,343.674462
7,0.836406,0.008223,0.313045,0.007240,0.327973,0.032783,4.0,sigmoid,NaN,"{'svr__C': 0.32797347187156445, 'svr__coef0': ...",-117908.398067,-1.192977e+05,-117131.202677,-1.181124e+05,8.961616e+02,6,343.674898
5,0.824755,0.028356,0.312754,0.006997,0.185086,0.158488,3.0,sigmoid,NaN,"{'svr__C': 0.18508616841607617, 'svr__coef0': ...",-117960.658677,-1.193459e+05,-117176.203279,-1.181609e+05,8.970391e+02,7,343.745448
4,0.865774,0.015379,0.332680,0.003206,0.202211,0.605793,3.0,sigmoid,NaN,"{'svr__C': 0.2022109779482142, 'svr__coef0': 0...",-117972.226520,-1.193566e+05,-117186.187626,-1.181717e+05,8.972126e+02,8,343.761059
2,0.705332,0.061179,0.364704,0.026387,0.367894,NaN,NaN,rbf,0.883131,"{'svr__C': 0.36789365022405685, 'svr__gamma': ...",-118020.203238,-1.194014e+05,-117227.389490,-1.182163e+05,8.983257e+02,9,343.826040
0,0.626445,0.032319,0.202219,0.024044,0.452325,0.097953,3.0,poly,NaN,"{'svr__C': 0.4523251407491419, 'svr__coef0': 0...",-117853.503550,-2.368237e+08,-117073.678358,-7.901953e+07,1.115844e+08,10,8889.293078


## Peform feature selection(Exercise 3)

In [ ]:
rnd_search.get_params(False)

{'cv': 3,
 'error_score': nan,
 'estimator': Pipeline(steps=[('preprocessing',
                  ColumnTransformer(remainder=Pipeline(steps=[('simpleimputer',
                                                               SimpleImputer(strategy='median')),
                                                              ('standardscaler',
                                                               StandardScaler())]),
                                    transformers=[('bedrooms',
                                                   Pipeline(steps=[('simpleimputer',
                                                                    SimpleImputer(strategy='median')),
                                                                   ('functiontransformer',
                                                                    FunctionTransformer(feature_names_out=<function ratio_name at 0x787b836...
                                                    'total_rooms', 'population',
            

In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

select_pipeline = Pipeline([
    ('preprocessing', preprocessing),
    ('feature_selection', SelectFromModel(RandomForestRegressor(random_state=42),
                                          threshold=0.005)),
    ('svr', SVR(C=rnd_search.best_params_["svr__C"],
                kernel=rnd_search.best_params_["svr__kernel"]))
])


In [ ]:
selector_rmses =  -cross_val_score(select_pipeline,
                                  train_set,
                                  train_labels,
                                  scoring="neg_root_mean_squared_error",
                                  cv=3)
pd.Series(selector_rmses).describe()

,0
count,3.000000
mean,115654.399596
std,1069.454143
min,114694.781664
25%,115077.938197
50%,115461.094731
75%,116134.208563
max,116807.322394


## Create a custom transformer for knn(Exercise 4)


In [14]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_array, check_is_fitted
from sklearn.base import clone


class KNeighborsTransformer(BaseEstimator, TransformerMixin):

  def __init__(self, n_neighbors=3):
    self.n_neighbors = n_neighbors # Store n_neighbors as an attribute
    self.knn_regressor = KNeighborsRegressor(n_neighbors=n_neighbors)


  def fit(self, X, y=None):
    self.n_features_in_ = X.shape[1]
    self.knn_regressor.fit(X, y)
    return self


  def transform(self, X):
    check_is_fitted(self)
    check_array(X)
    assert self.n_features_in_ == X.shape[1]
    return pd.DataFrame(self.knn_regressor.predict(X), columns=["knn_prediction"])


# won't work, since it is meant for stateless transformers.
# knn_transformer = FunctionTransformer(knnr, inverse_func=knnr.inverse_transform)
# knn_loc = knn_transformer.transform(train_set[["latitude", "longitude"]])

knn_transformer = KNeighborsTransformer()
knn_loc = knn_transformer.fit_transform(train_set[["latitude", "longitude"]], train_set['median_income'])

transformers = [(name, clone(transformer), columns)
                for name, transformer, columns in preprocessing.transformers]


geo_index = [name for name, _, _ in transformers].index("geo")


transformers[geo_index] = ("geo", knn_transformer,
                           ["latitude", "longitude", "median_income"])

new_geo_preprocessing = ColumnTransformer(transformers)
geo_pipeline = Pipeline([
    ('preprocessing', new_geo_preprocessing),
    ('svr', SVR(C=rnd_search.best_params_["svr__C"],
                kernel=rnd_search.best_params_["svr__kernel"]))
])

geo_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('svr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('bedrooms', ...), ('rooms_per_house', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the diff

In [14]:
from sklearn.model_selection import cross_val_score

new_pipe_rmses = -cross_val_score(geo_pipeline,
                                  train_set,
                                  train_labels,
                                  scoring="neg_root_mean_squared_error",
                                  cv=3)
pd.Series(new_pipe_rmses).describe()

count        3.000000
mean     70222.557529
std       4943.928872
min      64718.481700
25%      68190.547976
50%      71662.614252
75%      72974.595443
max      74286.576634
dtype: float64

## Random serach on KNNR (Exercise 5)


In [ ]:
from scipy.stats import expon, loguniform
param_distribs = {
    "preprocessing__geo__n_neighbors": range(1, 30),
    "svr__C": loguniform(20, 200_000),
    "svr__gamma": expon(scale=1.0),
}



new_geo_rnd_search = RandomizedSearchCV(geo_pipeline,
                                        param_distributions=param_distribs,
                                        n_iter=10,
                                          cv=3,
                                        scoring='neg_root_mean_squared_error',
                                        random_state=42)

new_geo_rnd_search.fit(train_set, train_labels)

In [ ]:
new_geo_rnd_search

## StandardScalerClone(Exercise 6)

In [ ]:
# class KNeighborsTransformer(BaseEstimator, TransformerMixin):

class StandardScalerClone(BaseEstimator):
    def 